In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


#### Question1 - 
- ##### We have a table which stores data of multiple sections. 
- ##### Every section has 3 numbers we have to find top 4 numbers from any 2 sections(2 numbers each) whose addition should be maximum. 
- ##### So in this case we will choose section b where we have 19(10+9) then we need to choose either C or D because both has sum of 18 but in D we have 10 which is big from 9 so we will give priority to D.

In [0]:
from pyspark.sql.types import *
data = [
    ("A", 5),
    ("A", 7),
    ("A", 10),
    ("B", 7),
    ("B", 9),
    ("B", 10),
    ("C", 9),
    ("C", 7),
    ("C", 9),
    ("D", 10),
    ("D", 3),
    ("D", 8)
]
schema = StructType([
    StructField("section", StringType(), True),
    StructField("number", IntegerType(), True)
])
section_data_df = spark.createDataFrame(data, schema)
section_data_df.write.mode("overwrite").saveAsTable("section_data")
section_data_df.display()

section,number
A,5
A,7
A,10
B,7
B,9
B,10
C,9
C,7
C,9
D,10


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = section_data_df.withColumn(
    "rnk",
    row_number().over(Window.partitionBy(col("section")).orderBy(col("number").desc()))
).filter(
    col("rnk") <= 2
).withColumn(
    "sum",
    sum(col("number")).over(Window.partitionBy(col("section")))
).withColumn(
    "max_per_section",
    max(col("number")).over(Window.partitionBy(col("section")))
).withColumn(
    "dnk",
    dense_rank().over(Window.orderBy(col("sum").desc(), col("max_per_section").desc()))
).filter(
    col("dnk") <= 2
)
    
df.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


section,number,rnk,sum,max_per_section,dnk
B,10,1,19,10,1
B,9,2,19,10,1
D,10,1,18,10,2
D,8,2,18,10,2
